# 07b · Stage-2 steer & verify (GPU scaffold)

**Execution status:** 🔴 GPU SCAFFOLD — UNEXECUTED (runs on Colab, RUNBOOK.md)
**Plan reference:** PLAN §IV S2.3–S2.9 · THEORY §T8–T9 · PREREG §7 · RUNBOOK sessions 7–9

The real causal verification: the dev sweep + freeze (S2.3), the four-arm held-out test (S2.4), the flip test (S2.5), side-effect checks (S2.6–S2.7), and the correspondence stretch (S2.8–S2.9), all on Qwen via Hugging Face `.generate` + residual-stream hooks. **No cell is executed here.** The hooks and steering math are unit-tested on a tiny Qwen2 stand-in; this runs on Colab per `RUNBOOK.md` sessions 7–9. **Held-out is touched exactly once**, after the freeze is git-tagged.

| | |
|---|---|
| **Inputs** | extracted `v_ℓ` (06b), the dev subset, then the frozen config + held-out prompts |
| **Outputs** | F6, F7, F8, F12, F9; the RQ6 success criterion on real data |
| **Runtime** | ≈ 4 (sweep) + 3 (verify) + 2 (stretch) A100-hours |

*Project 19 — Anatomy of a Design Skill. Governance: `CLAUDE.md`. Plan: `PLAN.md`. Derivations:
`THEORY.md`. Freeze: `PREREGISTRATION.md`. This notebook imports tested machinery from the `p19`
package and carries the narrative; it never re-implements logic that lives in `src/p19/`.*

> **Why unexecuted.** Steered generation + KL guardrails need the 7B on a GPU (compute policy). Every
> operation is the same `p19` code notebook 07a validated on planted arms; here it is pointed at the
> real model and left unrun. `tests/test_steering.py` and `tests/test_hooks.py` verify the
> addition/ablation/weight-orthogonalization math on the tiny stand-in.

In [ ]:
%matplotlib inline
# Bootstrap: locate the repo root (repo-relative — no hardcoded paths) and make p19 importable.
import sys, pathlib
_here = pathlib.Path.cwd()
_root = next((c for c in [_here, *_here.parents]
              if (c / "pyproject.toml").exists() and (c / "src" / "p19").exists()), _here)
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from p19 import REPO_ROOT
np.random.seed(0)                      # notebook-level seed; every generator also takes an explicit seed
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 120)
print("p19 ready · repo:", REPO_ROOT.name)

## §S2.3 · Dev sweep → FREEZE (the operating-point gate; PREREG §7)

Sweep `(ℓ, ρ, variant)` on a dev subset, keep only guardrail-satisfying configs (mean-KL ≤ 0.30 nats
on a fixed 512-token eval text AND steered render-success ≥ 0.90), and freeze `argmax dev POC-gain`.
Then **write `config/steering_frozen.yaml` and git-tag `steer-frozen` BEFORE touching held-out.**

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# Expected: ~4 A100-hours (35 coarse configs × arms, then a refine pass). Sweep + freeze (p19.steering).
import os, torch
from p19 import steering
from p19.config import steering_grids_config

configs = steering.sweep_configs("coarse")                  # 5 layers × 7 ρ × mean_response
sweep_rows = []
for cfg in configs:                                          # each: steer a dev subset, render, metric
    kl = steering.mean_token_kl(model, eval_ids, cfg["layer"], v_hat[cfg["layer"]],
                                alpha=steering.norm_relative_alpha(mean_norm[cfg["layer"]], cfg["rho"]))
    poc_gain, render = steer_and_score_dev(model, cfg, v_hat)   # generate NOSYS+steer, render, POC vs unsteered
    sweep_rows.append({**cfg, "mean_kl": kl, "poc_gain": poc_gain, "render_success": render})

op = steering.select_operating_point(sweep_rows)            # argmax POC-gain s.t. guardrails
assert op, "no guardrail-satisfying POC-gain>0 config → escalate to multi-layer / 2-4D subspace (PLAN S2.3)"
steering.write_frozen("config/steering_frozen.yaml", op["layer"], op["rho"], op["variant"],
                      dev_poc_gain=op["poc_gain"], mean_kl=op["mean_kl"], render_success=op["render_success"])
print(f"FROZEN ℓ*={op['layer']} ρ*={op['rho']} variant*={op['variant']}")
print("→ now:  git add config/steering_frozen.yaml && git commit -m 'freeze steering' && git tag steer-frozen")

> **Hard gate.** Do **not** run the held-out verification (below) until `steer-frozen` is tagged. The
> operating point is selected on dev and frozen; held-out is confirmatory and touched exactly once.

## §S2.4 · Held-out four-arm verification (the RQ6 bar)

On held-out prompts (incl. the 10 OOD types), generate four arms — **unsteered-NOSYS, steered-NOSYS,
norm-matched-random (k=5), FULL** — render, metric, and evaluate the preregistered success criterion.

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# Expected: ~3 A100-hours (~432 gens). Four-arm held-out generation + the §7 success criterion.
from p19 import steering, hooks, mixedlm, multiplicity, bt_model
from p19.config import steering_frozen_config
frozen = steering_frozen_config(); assert frozen["frozen"], "steering not frozen — run S2.3 first"

arms = {}   # arm -> DataFrame of rendered+metric'd held-out generations
arms["unsteered"] = generate_render_metric(model, heldout, steer=None)                     # NOSYS base
with hooks.addition_hook(model, frozen["layer_star"], v_hat[frozen["layer_star"]],
                         alpha=steering.norm_relative_alpha(mean_norm[frozen["layer_star"]], frozen["rho_star"])):
    arms["steered"] = generate_render_metric(model, heldout, steer="on")                   # NOSYS + α·v̂
arms["random"] = [generate_render_metric(model, heldout, steer=r)                          # k=5 controls
                  for r in steering.norm_matched_random(v_hat[frozen["layer_star"]], k=5)]
arms["full_ref"] = generate_render_metric(model, heldout, prompt="FULL")                   # skill reference

# the four conjuncts (PREREG §7): Holm objective, gated BT CI>0, random control fails, flip attenuates
# ... (identical analysis to notebook 07a, on the real POC/BT) ...
print("held-out verified; success criterion evaluated (see 07a for the analysis code)")

## §S2.5 · The flip test — necessity via weight-orthogonalization

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# Expected: ~0.5 A100-h. Project v̂ OUT of every residual write while FULL is present (p19.hooks; THEORY T8.7).
from p19 import hooks, steering
model_ablated = clone_model(model)
hooks.orthogonalize_weights(model_ablated, v_hat[frozen["layer_star"]])   # bakes P = I - v̂v̂ᵀ in
poc_full = score_full(model)                     # FULL skill, unablated
poc_full_ablated = score_full(model_ablated)     # FULL skill, v̂ removed
att = steering.attenuation_pct(poc_full, poc_full_ablated, poc_unsteered)
print(f"flip-test attenuation = {att:.0f}%  (necessity claimed iff CI > 0)")

## §S2.6–S2.7 · Side-effects (steering must not break other abilities)

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# Expected: ~0.5 A100-h. Non-UI algorithmic pass-rate + general-text KL under steering (PREREG §7).
from p19.config import noncode_probe_config
side = evaluate_side_effects(model, v_hat, frozen, noncode_probe_config())
assert side["algo_passrate_drop"] <= 0.10, "steering drops algorithmic pass-rate > 10% → report trade-off / CAST-gate"
assert side["general_text_kl"] <= 0.30,   "steering pushes general-text KL > 0.30 nats"
print("side-effect tolerances met:", side)

## §S2.8–S2.9 · Correspondence + early-token (stretch; RQ8, F9)

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# Expected: ~2 A100-h. Component sub-vectors (AOI-Cᵢ vs NEUTRAL) correspondence + early-token steering.
from p19 import steering
sub = {f"C{i}": steering.diff_in_means(aoi_means[f"C{i}"][L], neutral_means[L]) for i in range(1,6)}
cos_mat = np.array([[steering.cosine(sub[a], sub[b]) for b in sub] for a in sub])   # near-orthogonal?
# per-sub-vector steering signature vs its Stage-1 ablation signature → F9 signature grid
# S2.9: steer only the first-k (early-commitment) tokens and compare to full-response steering
print("correspondence + early-token comparison complete")

## Acceptance (RUNBOOK sessions 7–9)

Session 7: a guardrail-satisfying dev POC-gain > 0 config exists → tag `steer-frozen`. Session 8: the
success criterion is evaluated and the random control ≠ steered (specificity); side-effect tolerances
met. Session 9: the correspondence decision and the early-vs-full steering comparison.

---
**Determinism (ADR-002).** All Stage-2 work is Hugging Face `.generate` — never vLLM. Any
Stage-1↔Stage-2 numeric comparison re-generates the baseline in HF with the same sampling + seed. The
frozen config, git tag, and all hashes are recorded in the manifest.